# 32. Cost Engineering

**Tier:** Production & Safety
**Estimated time:** 45 minutes
**Prerequisites:** 13, 16, 29
**Priority:** 🟡 Important — cost is what gets AI features killed in production; routing, caching, and batching routinely cut bills 5-10x, and the engineer who owns cost owns the roadmap conversation. *If skipped, revisit when:* the first invoice that makes someone wince, or before any scale-up decision.
**Source material:** @akshay_pachaar RAG vs CAG (caching economics) — https://x.com/akshay_pachaar/status/2056714042455343160 ; notebook 13 (KV cache), notebook 16 (RAG vs CAG)

## What You'll Learn
- Model routing: try a cheap model first, escalate to an expensive one only on failure
- Prompt-caching strategy at scale, extending notebook 16's cache-hit-rate concept
- Batch API economics — when trading latency for a steep cost discount makes sense
- Per-feature token budgets as a first-class design constraint, not an afterthought

## Why This Matters
A feature that's correct and fast but costs 10x what the business can sustain gets killed just as surely as one that's broken. Cost engineering turns "make it cheaper" from a vague plea into a measured, structural decision: which requests need the expensive model, which content is cacheable, which work can wait for a batch discount. This is the notebook that lets you own the cost conversation with evidence instead of guesses.


In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
CHEAP_MODEL = "claude-haiku-4-5-20251001"
EXPENSIVE_MODEL = "claude-haiku-4-5-20251001"   # stand-in for a stronger/pricier model in this teaching environment

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — live routing/cost cells will be skipped.")


## Model routing — cheap first, escalate on failure

Most requests are easy; a minority are hard. Routing every request to the most expensive model wastes money on the easy majority; routing everything to the cheapest model fails the hard minority. The fix: try the cheap model, detect low-confidence or failed responses, and escalate ONLY those to the expensive model — paying the premium only where it's earned.

In [ ]:
# Approximate per-million-token pricing (USD) — check current pricing before using in prod.
PRICE = {
    "cheap": {"input": 1.00, "output": 5.00},
    "expensive": {"input": 3.00, "output": 15.00},
}

def estimate_cost(tier, in_tok, out_tok):
    p = PRICE[tier]
    return (in_tok / 1e6) * p["input"] + (out_tok / 1e6) * p["output"]

def needs_escalation(answer):
    """A cheap confidence heuristic: hedging language signals the cheap model wasn't sure."""
    hedge_words = ["i'm not sure", "might be", "possibly", "i don't know", "unclear"]
    return any(w in answer.lower() for w in hedge_words)

def routed_ask(prompt, max_tokens=100):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]", "cheap", 0.0
    cheap_resp = client.messages.create(model=CHEAP_MODEL, max_tokens=max_tokens,
                                         messages=[{"role": "user", "content": prompt}])
    answer = cheap_resp.content[0].text
    in_tok, out_tok = cheap_resp.usage.input_tokens, cheap_resp.usage.output_tokens
    cost = estimate_cost("cheap", in_tok, out_tok)
    if needs_escalation(answer):
        expensive_resp = client.messages.create(model=EXPENSIVE_MODEL, max_tokens=max_tokens,
                                                 messages=[{"role": "user", "content": prompt}])
        answer = expensive_resp.content[0].text
        cost += estimate_cost("expensive", expensive_resp.usage.input_tokens, expensive_resp.usage.output_tokens)
        return answer, "escalated", cost
    return answer, "cheap", cost

queries = [
    "What is the capital of France?",
    "In one sentence, what's a plausible reason a distributed cache might return stale data?",
    "What is 7 + 5?",
]
total_cost = 0.0
for q in queries:
    answer, tier, cost = routed_ask(q)
    total_cost += cost
    print(f"[{tier:>9s}] ${cost:.6f}  Q={q!r}\n            -> {answer[:80]!r}")
print(f"\nTotal cost across {len(queries)} routed requests: ${total_cost:.6f}")


## Prompt caching at scale

Notebook 16 introduced RAG vs. CAG and the idea of a cache hit-rate. At scale, prompt caching (reusing the KV-cache state for a repeated prefix — see notebook 13) turns a large "static" portion of your prompt (system instructions, a knowledge base, few-shot examples) into a near-free constant cost, paid once per cache TTL window instead of on every single request. The savings compound directly with request volume.

In [ ]:
STATIC_PREFIX_TOKENS = 2000    # system prompt + reference docs, repeated on every request
DYNAMIC_SUFFIX_TOKENS = 150     # the actual user query, different every time
CACHE_DISCOUNT = 0.9            # cached input tokens cost ~10% of normal (illustrative figure)

def cost_without_caching(n_requests):
    per_request = estimate_cost("cheap", STATIC_PREFIX_TOKENS + DYNAMIC_SUFFIX_TOKENS, 100)
    return per_request * n_requests

def cost_with_caching(n_requests):
    first_request = estimate_cost("cheap", STATIC_PREFIX_TOKENS + DYNAMIC_SUFFIX_TOKENS, 100)
    cached_prefix_cost = estimate_cost("cheap", STATIC_PREFIX_TOKENS, 0) * (1 - CACHE_DISCOUNT)
    dynamic_cost = estimate_cost("cheap", DYNAMIC_SUFFIX_TOKENS, 100)
    subsequent_requests = cached_prefix_cost + dynamic_cost
    return first_request + subsequent_requests * (n_requests - 1)

volumes = [1, 10, 100, 1000, 10000]
for n in volumes:
    no_cache = cost_without_caching(n)
    with_cache = cost_with_caching(n)
    savings_pct = (1 - with_cache / no_cache) * 100
    print(f"n={n:>6}  no-cache=${no_cache:8.4f}  cached=${with_cache:8.4f}  savings={savings_pct:5.1f}%")


In [ ]:
%matplotlib inline
n_range = np.array([1, 10, 50, 100, 500, 1000, 5000, 10000])
no_cache_costs = [cost_without_caching(n) for n in n_range]
cached_costs = [cost_with_caching(n) for n in n_range]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_range, no_cache_costs, marker="o", label="no caching")
ax.plot(n_range, cached_costs, marker="o", label="with prompt caching")
ax.set_xscale("log")
ax.set_xlabel("number of requests (log scale)")
ax.set_ylabel("total cost (USD)")
ax.set_title("Prompt caching: savings compound with request volume")
ax.legend()
plt.tight_layout()
plt.show()


*The gap between the two curves widens as volume grows — caching turns a large repeated prefix from a per-request cost into a near-fixed cost, which is exactly why notebook 16's "92% cache hit-rate" claim matters more at production scale than in a demo.*

## Batch API economics

Some work has no user staring at a spinner: nightly summarization, bulk classification, embedding a large corpus. For that work, batch APIs trade latency (results arrive within hours, not seconds) for a steep discount — commonly around 50% off standard pricing. The decision is a simple one once you frame it correctly: does this task have a human waiting synchronously, or can it tolerate an hours-long turnaround?

In [ ]:
BATCH_DISCOUNT = 0.5   # illustrative — batch APIs commonly run ~50% of synchronous pricing

def batch_vs_sync_cost(n_items, in_tok=200, out_tok=100):
    sync_cost = estimate_cost("cheap", in_tok, out_tok) * n_items
    batch_cost = sync_cost * BATCH_DISCOUNT
    return sync_cost, batch_cost

for n_items in [100, 10_000, 1_000_000]:
    sync_cost, batch_cost = batch_vs_sync_cost(n_items)
    print(f"{n_items:>9,} items:  sync=${sync_cost:10,.2f}   batch=${batch_cost:10,.2f}   saved=${sync_cost - batch_cost:10,.2f}")

print("\nDecision rule: if nothing is synchronously waiting on the result, batch it — the")
print("discount is 'free' money left on the table otherwise.")


## Per-feature token budgets

Treat a token budget as a design constraint from day one, the same way you'd budget latency or memory. A budget forces explicit tradeoffs — truncate context, summarize history, or downgrade model tier — instead of discovering the cost problem after shipping.

In [ ]:
FEATURE_BUDGETS = {
    "chat_reply": {"max_tokens_per_request": 500, "max_requests_per_user_per_day": 200},
    "document_summarization": {"max_tokens_per_request": 4000, "max_requests_per_user_per_day": 20},
    "background_classification": {"max_tokens_per_request": 50, "max_requests_per_user_per_day": 100_000},
}

def daily_cost_ceiling(feature, tier="cheap"):
    budget = FEATURE_BUDGETS[feature]
    per_request = estimate_cost(tier, budget["max_tokens_per_request"], budget["max_tokens_per_request"] // 4)
    return per_request * budget["max_requests_per_user_per_day"]

for feature in FEATURE_BUDGETS:
    ceiling = daily_cost_ceiling(feature)
    print(f"{feature:28s} worst-case daily cost per user: ${ceiling:.4f}")


## Exercises

**Exercise 1 (Warm-up):** Change `needs_escalation`'s hedge-word list to also flag answers under 3 words as "possibly too terse to be useful," and re-run `routed_ask` on the three queries. Does the escalation rate change?

**Exercise 2 (Apply):** Implement `optimal_cache_ttl(request_rate_per_hour, cache_ttl_options=[5, 30, 60])` that estimates, for each candidate TTL (minutes), the effective cache hit-rate assuming Poisson-arriving requests, and returns the TTL with the best cost/hit-rate tradeoff.

**Exercise 3 (Extend):** Notebook 29 built a served FastAPI app with a primary/fallback pattern. Sketch how you'd merge that pattern with this notebook's model routing so the SAME `try/except` structure handles both "primary model is down" (notebook 29) and "primary model wasn't confident enough" (this notebook) with one unified escalation path.


In [ ]:
# Exercise 1: Warm-up
# Task: Extend needs_escalation to also flag answers under 3 words, and re-run routed_ask.
# Hint: len(answer.split()) < 3 is the check; combine with `or` against the existing hedge check.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement optimal_cache_ttl(request_rate_per_hour, cache_ttl_options) -> best_ttl.
# Hint: a longer TTL raises hit-rate but risks serving stale content; model hit-rate as
# 1 - exp(-request_rate_per_hour * ttl_hours) and weigh it against a staleness cost you define.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch a unified try/except that escalates on EITHER a provider failure or low confidence.
# Hint: two separate conditions can share one escalation branch: `except Exception` OR
# `if needs_escalation(answer)`, both leading to the same expensive-model call.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
def needs_escalation_v2(answer):
    hedge_words = ["i'm not sure", "might be", "possibly", "i don't know", "unclear"]
    too_terse = len(answer.split()) < 3
    return any(w in answer.lower() for w in hedge_words) or too_terse

# Exercise 2
def optimal_cache_ttl(request_rate_per_hour, cache_ttl_options=(5, 30, 60)):
    best_ttl, best_score = None, -1
    for ttl_min in cache_ttl_options:
        ttl_hours = ttl_min / 60
        hit_rate = 1 - np.exp(-request_rate_per_hour * ttl_hours)
        staleness_penalty = ttl_hours * 0.1   # illustrative: longer TTL = more staleness risk
        score = hit_rate - staleness_penalty
        if score > best_score:
            best_ttl, best_score = ttl_min, score
    return best_ttl

print(optimal_cache_ttl(50))

# Exercise 3
def unified_routed_ask(prompt, max_tokens=100):
    try:
        resp = client.messages.create(model=CHEAP_MODEL, max_tokens=max_tokens,
                                       messages=[{"role": "user", "content": prompt}])
        answer = resp.content[0].text
        if needs_escalation(answer):
            raise ValueError("low confidence, escalating")
    except Exception:
        resp = client.messages.create(model=EXPENSIVE_MODEL, max_tokens=max_tokens,
                                       messages=[{"role": "user", "content": prompt}])
        answer = resp.content[0].text
    return answer
# Both a hard failure (network, rate limit, timeout — notebook 29) and a soft failure (low
# confidence — this notebook) now route through the same escalation branch.
```
</details>

## Key Takeaways
- Model routing (cheap first, escalate on failure/low-confidence) pays the expensive-model premium only on the minority of requests that actually need it.
- Prompt caching turns a large repeated prefix into a near-fixed cost — savings compound with request volume, which is why notebook 16's cache hit-rate matters far more in production than in a demo.
- Batch APIs trade latency for a steep discount — the deciding question is simply "is anyone synchronously waiting on this result?"
- Treat token budgets as a per-feature design constraint from day one, not a number you discover after the first expensive invoice.
- Notebook 29's primary/fallback pattern and this notebook's routing pattern are the same `try/except` shape — a provider failure and a low-confidence answer can share one escalation path.

## What's Next
Notebook 33 wires the eval harness (notebook 24) into CI — the regression gate that catches a prompt or routing change before it ships, instead of after.
